In [47]:
import requests
from bs4 import BeautifulSoup
import re
import pandas as pd

In [21]:
!pip install lxml
import sys
!{sys.executable} -m pip install lxml


[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip
'c:\Users\krishna' is not recognized as an internal or external command,
operable program or batch file.


In [22]:
url = 'https://www.indiafilings.com/gst'
response = requests.get(url)
response

<Response [403]>

In [23]:
data = response.text
for i in data:
    print(i, end="")

<!DOCTYPE html><html lang="en-US"><head><title>Just a moment...</title><meta http-equiv="Content-Type" content="text/html; charset=UTF-8"><meta http-equiv="X-UA-Compatible" content="IE=Edge"><meta name="robots" content="noindex,nofollow"><meta name="viewport" content="width=device-width,initial-scale=1"><style>*{box-sizing:border-box;margin:0;padding:0}html{line-height:1.15;-webkit-text-size-adjust:100%;color:#313131;font-family:system-ui,-apple-system,BlinkMacSystemFont,"Segoe UI",Roboto,"Helvetica Neue",Arial,"Noto Sans",sans-serif,"Apple Color Emoji","Segoe UI Emoji","Segoe UI Symbol","Noto Color Emoji"}body{display:flex;flex-direction:column;height:100vh;min-height:100vh}.main-content{margin:8rem auto;padding-left:1.5rem;max-width:60rem}@media (width <= 720px){.main-content{margin-top:4rem}}#challenge-error-text{background-image:url("");background-repeat:no-repeat;background-size:contain;padding-left:34px}</style><meta http-equiv="refresh" content="360"></head><body><div class="mai

In [24]:
from bs4 import BeautifulSoup

html = "<html><body><h1>Hello Krishna</h1></body></html>"

soup = BeautifulSoup(html, "lxml")

print(soup.h1.text)

Hello Krishna


In [ ]:
soup = BeautifulSoup(data, "lxml")

print(soup.prettify())

In [26]:
text = soup.get_text()
print(text)

Just a moment...Enable JavaScript and cookies to continue


In [31]:
print(len(soup.find_all("h1")))

0


In [38]:
file_path = "Data/gst.html"

with open(file_path, "r", encoding="utf-8") as f:
    html = f.read()
soup = BeautifulSoup(html, "lxml")
text = soup.get_text()

In [39]:
for tag in soup.find_all(["h1","h2","h3"]):
    text = tag.get_text(strip=True)
    if text:
        print(text)
    else:
        print("No text found in this tag.")

Online GST Registration
Apply for GST
Our Clients
Why GST Registration is Essential?
Why Choose IndiaFilings?
Simple packages. Transparentpricing.
GST Registration – GST Registration + Monthly Filing
GST Return Filing – 1 Year GST Filing
IndiaFilings - Accountant – 1 Year GST Filing + ITR
Popular Searches
IndiaFilings Search – Quick and Precise Results!


In [56]:
price_pattern = re.compile(r"(₹\s?\d[\d,]*|Rs\.?\s?\d[\d,]*)")

In [43]:
results = []

for tag in soup.find_all(True):   # True = all tags
    text = tag.get_text(strip=True)

    matches = price_pattern.findall(text)

    for price in matches:
        results.append({
            "tag": tag.name,
            "price": price,
            "text_context": text[:150]   # store some context
        })

In [53]:
results

[{'tag': 'html',
  'price': '₹1499',
  'text_context': "Goods & Service Tax Registration, Filing & Software ServicesIndiaFilingsStartupRegistrationsTrademarkGSTIncome TaxMCAComplianceGlobalWe're Hiring! 🥳×s"},
 {'tag': 'html',
  'price': '₹1499',
  'text_context': "Goods & Service Tax Registration, Filing & Software ServicesIndiaFilingsStartupRegistrationsTrademarkGSTIncome TaxMCAComplianceGlobalWe're Hiring! 🥳×s"},
 {'tag': 'body',
  'price': '₹1499',
  'text_context': "IndiaFilingsStartupRegistrationsTrademarkGSTIncome TaxMCAComplianceGlobalWe're Hiring! 🥳×searchWebImageSort byRelevanceDateLoginOnline GST Registratio"},
 {'tag': 'body',
  'price': '₹1499',
  'text_context': "IndiaFilingsStartupRegistrationsTrademarkGSTIncome TaxMCAComplianceGlobalWe're Hiring! 🥳×searchWebImageSort byRelevanceDateLoginOnline GST Registratio"},
 {'tag': 'section',
  'price': '₹1499',
  'text_context': 'Online GST RegistrationRegister for GST online with dedicated experts from IndiaFilings. We take care

In [52]:
df = pd.DataFrame(results)

print(df['text_context'].iloc[1])


Goods & Service Tax Registration, Filing & Software ServicesIndiaFilingsStartupRegistrationsTrademarkGSTIncome TaxMCAComplianceGlobalWe're Hiring! 🥳×s


In [60]:
import json
from bs4 import BeautifulSoup

def extract_html_to_dict(file_path):
    # Load the local HTML file
    with open(file_path, 'r', encoding='utf-8') as f:
        html_content = f.read()

    soup = BeautifulSoup(html_content, 'lxml')
    
    # 1. Extract Metadata
    data = {
        "metadata": {
            "title": soup.title.string if soup.title else "No Title",
            "description": soup.find("meta", attrs={"name": "description"})["content"] if soup.find("meta", attrs={"name": "description"}) else ""
        },
        "content": []
    }

    # 2. Extract Headings and Paragraphs (Contextual Flow)
    # We iterate through tags that usually contain the "meat" of the page
    for tag in soup.find_all(['h1', 'h2', 'h3', 'p', 'table']):
        entry = {"type": tag.name}
        
        if tag.name == 'table':
            # Handle tables separately to keep data structured
            rows = []
            for tr in tag.find_all('tr'):
                cells = [td.get_text(strip=True) for td in tr.find_all(['td', 'th'])]
                rows.append(cells)
            entry["value"] = rows
        else:
            # Clean up whitespace and newlines for text tags
            text = tag.get_text(separator=" ", strip=True)
            if text:
                entry["value"] = text
        
        if "value" in entry:
            data["content"].append(entry)

    return data

# Usage
file_name = "C:\\Users\\krishna varathan\\Desktop\\web_scrapping\\Data\\indiafilings.html" # Replace with your local file path
extracted_data = extract_html_to_dict(file_name)

# Step 3: Save and View as JSON
with open('extracted_data.json', 'w', encoding='utf-8') as json_file:
    json.dump(extracted_data, json_file, indent=4, ensure_ascii=False)

print("Extraction Complete! Check 'extracted_data.json'.")

Extraction Complete! Check 'extracted_data.json'.
